# `wf_set_account_alias` 단계별 Testbed

이 Notebook은 계좌 별칭 변경 Workflow를 한 번에 실행하고 결과만 보는 용도가
아니다. 각 단계(Slot 추출 → 계좌 확인 → 별칭 값 확인 → Prepare → 승인 → Execute →
결과)에서 Agent가 실제로 주고받는 요청/응답과 중단 State를 하나씩 확인한다.

## 전체 실행 흐름

```text
사용자 발화 ("생활비 통장 별칭을 여행 자금으로 바꿔줘")
  → [Agent 내부] 계좌 힌트·별칭 추출
  → [Agent → Backend API] 계좌 확인(fetch_accounts, account_capability=settings)
  → [Agent 내부] 별칭 값 확인 (있으면 바로 Prepare, 없으면 입력 화면)
  → [Agent → Backend Webhook] 승인 화면 요청 (1차 중단)
  → 사용자 승인(approved)
  → [Agent → Backend API] Execute
  → [Agent → Backend Webhook] 결과 표시
```

`wf_set_default_account`와 달리 수정 대상이 "account"·"alias" 둘이라
`_RESET_STEP_BY_TARGET`으로 분기한다.

## 0. 공통 설정과 출력 함수

In [1]:
import json
import os

from pydantic import SecretStr

from agent.clients.backend import BackendClientConfig
from agent.runtime.hitl import ExecutionResumeRequest
from agent.testing import MockBackend
from agent.testing.set_account_alias import (
    create_account_alias_change_backend_testbed,
    create_account_alias_change_mock_testbed,
)
from agent.workflow_contracts import WorkflowContractStore
from agent.workflows.set_account_alias import extract_account_alias_slots_from_text


def show_json(title, value):
    print(chr(10) + f"--- {title} ---")
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


CHAT_SESSION_ID = "chat_testbed_account_alias"
EXECUTION_CONTEXT_ID = "exec_testbed_account_alias"
THREAD_ID = "thread_account_alias_testbed_001"

print("공통 설정 완료")

공통 설정 완료


## 1. 관리시트 계약에서 실행 Step 확인

Notebook에서 임의로 Workflow 순서를 정하지 않는다. 관리시트에서 생성한 Manifest를
읽어 현재 Step, 통신 방식과 다음 Route를 그대로 보여준다.

In [2]:
store = WorkflowContractStore()
workflow_contract = store.get_workflow("wf_set_account_alias")

step_contracts = [
    {
        "order": step["step_order"],
        "step_id": step["step_id"],
        "mode": step["interaction_mode"],
        "contract_id": step.get("contract_id"),
        "next": step["route_summary"],
    }
    for step in workflow_contract["steps"]
]
show_json("Workflow Step 계약", step_contracts)


--- Workflow Step 계약 ---
[
  {
    "order": 1,
    "step_id": "extract_account_alias_slots",
    "mode": "agent_internal",
    "contract_id": null,
    "next": "항상 → resolve_account_alias_target"
  },
  {
    "order": 2,
    "step_id": "resolve_account_alias_target",
    "mode": "backend_tool_api",
    "contract_id": "API-ACCOUNT-LIST",
    "next": "resolved → check_account_alias_value | selection_required → request_account_alias_selection | no_accounts → emit_account_alias_selection_empty | 오류 → emit_account_alias_error"
  },
  {
    "order": 3,
    "step_id": "request_account_alias_selection",
    "mode": "webhook_then_resume",
    "contract_id": "UI-ACCOUNT-ALIAS-SELECTION",
    "next": "selected → check_account_alias_value | cancelled → END | 계약 오류 → emit_account_alias_error"
  },
  {
    "order": 4,
    "step_id": "emit_account_alias_selection_empty",
    "mode": "webhook",
    "contract_id": "UI-ACCOUNT-ALIAS-SELECTION",
    "next": "항상 → END"
  },
  {
    "order": 5,
    "step_

## 2. Step 1 — 사용자 발화에서 Slot 추출

이 Step은 Agent 내부에서만 실행되며 Backend를 호출하지 않는다. 실제 실행 시에는
LLM 우선 추출기(`extract_account_alias_slots_llm_first`)가 쓰이지만, 이 Notebook은
결정적 결과를 보여주기 위해 규칙 기반 폴백 함수를 직접 부른다.

In [3]:
USER_MESSAGE = "생활비 통장 별칭을 여행 자금으로 바꿔줘"

slot_input = {"message": USER_MESSAGE}
slot_output = dict(extract_account_alias_slots_from_text(USER_MESSAGE))

show_json("Step 1 입력", slot_input)
show_json("Step 1 출력", slot_output)

assert slot_output == {"account_hint": "생활비 통장", "alias": "여행 자금"}


--- Step 1 입력 ---
{
  "message": "생활비 통장 별칭을 여행 자금으로 바꿔줘"
}

--- Step 1 출력 ---
{
  "account_hint": "생활비 통장",
  "alias": "여행 자금"
}


### 다른 발화를 직접 넣어보기

이 Cell은 Workflow 실행에 영향을 주지 않는 연습용이다. 별칭 없이 계좌만 말하면
`alias`가 `None`으로 남아 이후 입력 화면으로 이어진다.

In [4]:
practice_messages = [
    "생활비 통장 별칭을 바꾸고 싶어",
    "신한은행 계좌 별칭을 '여행 자금'으로 변경해줘",
]

for message in practice_messages:
    show_json(message, dict(extract_account_alias_slots_from_text(message)))


--- 생활비 통장 별칭을 바꾸고 싶어 ---
{
  "account_hint": "생활비 통장",
  "alias": null
}

--- 신한은행 계좌 별칭을 '여행 자금'으로 변경해줘 ---
{
  "account_hint": "신한은행 계좌",
  "alias": "여행 자금"
}


## 3. Mock Backend 입력값 준비

Backend가 아직 없어도 실제 API와 같은 응답 Schema로 Workflow를 실행한다.

In [5]:
def account(account_id, alias, bank_name="신한은행"):
    return {
        "account_id": account_id,
        "bank_name": bank_name,
        "account_alias": alias,
        "account_type": "checking",
        "masked_account_number": "110-***-123456",
        "currency": "KRW",
        "is_default": False,
        "status": "active",
    }


SELECTED_ACCOUNT_ID = "acc_001"
ACCOUNT_CANDIDATES = [account(SELECTED_ACCOUNT_ID, "생활비")]

### Prepare · 실행 단계에서 사용할 Mock 응답

In [6]:
CONFIRMATION_VIEW = {
    "account": {
        "account_id": SELECTED_ACCOUNT_ID,
        "bank_name": "신한은행",
        "masked_account_number": "110-***-123456",
    },
    "alias": "여행 자금",
    "expires_at": "2026-07-19T10:05:00+09:00",
}

## 4. Mock Backend와 실제 Workflow 연결

예상하지 않은 Method나 Path가 호출되면 Mock Backend가 즉시 실패한다.

In [7]:
mock_backend = MockBackend()
mock_backend.add_success(
    "GET",
    "/api/v1/agent-tools/accounts",
    {
        "account_resolution_outcome": "resolved",
        "accounts": ACCOUNT_CANDIDATES,
        "account_ids": [SELECTED_ACCOUNT_ID],
    },
)
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/settings/account-alias:prepare",
    {
        "outcome": "ready_for_confirmation",
        "confirmation_id": "confirm_alias_testbed_001",
        "confirmation_view": CONFIRMATION_VIEW,
    },
)
mock_backend.add_success(
    "POST", "/api/v1/webhooks/agent", {"message_id": "message_approval_001"}
)
mock_backend.add_success(
    "POST",
    "/api/v1/agent-tools/settings/account-alias",
    {
        "outcome": "completed",
        "account_id": SELECTED_ACCOUNT_ID,
        "alias": "여행 자금",
        "completed_at": "2026-07-19T10:06:00+09:00",
    },
)
mock_backend.add_success(
    "POST", "/api/v1/webhooks/agent", {"message_id": "message_result_001"}
)

mock_config = BackendClientConfig(
    base_url="http://backend.test",
    agent_service_token=SecretStr("agent-service-token"),
    agent_webhook_secret=SecretStr("agent-webhook-secret"),
    retry_backoff_seconds=0,
)

testbed = create_account_alias_change_mock_testbed(
    mock_backend, mock_config, thread_id=THREAD_ID
)
print("Mock Testbed 연결 완료")

Mock Testbed 연결 완료


## 5. Step 1·2·3 실행 — 계좌 확인·별칭 확인 후 승인 대기(1차 중단)

계좌 힌트가 정확히 하나로 확정되고 발화에 별칭도 이미 있으므로, 선택·입력 화면
없이 바로 Prepare를 거쳐 승인 화면에서 멈춘다.

In [8]:
start_request = {
    "message": USER_MESSAGE,
    "request_id": "req_alias_start_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
}
show_json("Workflow 시작 입력", start_request)

waiting_approval = await testbed.start(**start_request)
show_json("1차 중단 결과", waiting_approval.model_dump(mode="json"))

assert waiting_approval.status == "waiting"
assert waiting_approval.pending_interaction["type"] == "approval"
CONFIRMATION_ID = waiting_approval.pending_interaction["confirmation_id"]


--- Workflow 시작 입력 ---
{
  "message": "생활비 통장 별칭을 여행 자금으로 바꿔줘",
  "request_id": "req_alias_start_001",
  "chat_session_id": "chat_testbed_account_alias",
  "execution_context_id": "exec_testbed_account_alias"
}

--- 1차 중단 결과 ---
{
  "agent_thread_id": "thread_account_alias_testbed_001",
  "status": "waiting",
  "pending_interaction": {
    "type": "approval",
    "workflow_id": "wf_set_account_alias",
    "step_id": "request_account_alias_approval",
    "ui_contract_id": "UI-ACCOUNT-ALIAS-CONFIRMATION",
    "input_request_id": null,
    "confirmation_id": "confirm_alias_testbed_001",
    "auth_context_id": null
  },
  "webhook_message_id": "message_approval_001",
  "replayed": false
}


### 실제 계좌 확인 API 요청과 승인 UI Webhook

In [9]:
exchanges = mock_backend.exchange_timeline(include_payload=True)
show_json("Agent → Backend 계좌 확인 요청", exchanges[0]["request"])
show_json("Agent → Backend Prepare 요청", exchanges[1]["request"])
show_json("Agent → Backend 승인 화면 Webhook", exchanges[2]["request"])


--- Agent → Backend 계좌 확인 요청 ---
{
  "account_hint": "생활비 통장",
  "account_capability": "settings",
  "resolve_selection": "true",
  "all_accounts_requested": "false",
  "limit": "20"
}

--- Agent → Backend Prepare 요청 ---
{
  "account_id": "acc_001",
  "alias": "여행 자금"
}

--- Agent → Backend 승인 화면 Webhook ---
{
  "chat_session_id": "chat_testbed_account_alias",
  "event_type": "need_approval",
  "content": "계좌 별칭 변경 내용을 확인하고 승인해 주세요.",
  "confirmation_id": "confirm_alias_testbed_001",
  "metadata": {
    "workflow_id": "wf_set_account_alias",
    "step_id": "request_account_alias_approval",
    "ui_contract_id": "UI-ACCOUNT-ALIAS-CONFIRMATION",
    "ui": {
      "type": "confirm_modal",
      "payload": {
        "account": {
          "account_id": "acc_001",
          "bank_name": "신한은행",
          "masked_account_number": "110-***-123456"
        },
        "alias": "여행 자금",
        "expires_at": "2026-07-19T10:05:00+09:00",
        "actions": [
          "approve",
          "modif

## 6. Step 4 실행 — 승인 후 실행과 결과

승인(`approved`)으로 Resume하면 Agent는 Execute를 호출하고 결과 Webhook을 전송한
뒤 종료한다.

In [10]:
resume_approval_request = {
    "request_id": "req_alias_resume_approval_001",
    "chat_session_id": CHAT_SESSION_ID,
    "execution_context_id": EXECUTION_CONTEXT_ID,
    "resume": {
        "type": "approval",
        "confirmation_id": CONFIRMATION_ID,
        "approval_outcome": "approved",
    },
}
show_json("승인 Resume 입력", resume_approval_request)

completed = await testbed.resume(
    THREAD_ID, ExecutionResumeRequest.model_validate(resume_approval_request)
)
show_json("완료 결과", completed.model_dump(mode="json"))

assert completed.status == "completed"


--- 승인 Resume 입력 ---
{
  "request_id": "req_alias_resume_approval_001",
  "chat_session_id": "chat_testbed_account_alias",
  "execution_context_id": "exec_testbed_account_alias",
  "resume": {
    "type": "approval",
    "confirmation_id": "confirm_alias_testbed_001",
    "approval_outcome": "approved"
  }
}

--- 완료 결과 ---
{
  "agent_thread_id": "thread_account_alias_testbed_001",
  "status": "completed",
  "pending_interaction": null,
  "webhook_message_id": null,
  "replayed": false
}


### 완료 시점의 Agent State

계좌 ID·별칭·완료 시각이 모두 반영된다.

In [11]:
completed_state = await testbed.state(THREAD_ID)
show_json(
    "완료 시점 시스템 State",
    {
        "current_step_id": completed_state.get("current_step_id"),
        "route_key": completed_state.get("route_key"),
        "status": completed_state.get("status"),
    },
)
show_json("완료 시점 Workflow Data", completed_state["data"])
assert completed_state["data"]["account_id"] == SELECTED_ACCOUNT_ID
assert completed_state["data"]["alias"] == "여행 자금"
assert completed_state["data"]["completed_at"] is not None


--- 완료 시점 시스템 State ---
{
  "current_step_id": "emit_account_alias_result",
  "route_key": "completed",
  "status": "completed"
}

--- 완료 시점 Workflow Data ---
{
  "account_hint": "생활비 통장",
  "alias": "여행 자금",
  "account_resolution_outcome": "resolved",
  "accounts": [
    {
      "account_id": "acc_001",
      "bank_name": "신한은행",
      "account_alias": "생활비",
      "account_type": "checking",
      "masked_account_number": "110-***-123456",
      "currency": "KRW",
      "is_default": false,
      "status": "active"
    }
  ],
  "account_id": "acc_001",
  "prepare_attempt": 1,
  "correction_view": null,
  "confirmation_id": "confirm_alias_testbed_001",
  "confirmation_view": {
    "account": {
      "account_id": "acc_001",
      "bank_name": "신한은행",
      "masked_account_number": "110-***-123456"
    },
    "alias": "여행 자금",
    "expires_at": "2026-07-19T10:05:00+09:00"
  },
  "approval_outcome": "approved",
  "change_target": null,
  "completed_at": "2026-07-19T10:06:00+09:00"
}


## 7. 전체 통신 순서와 자동 확인

In [12]:
timeline = testbed.request_timeline()
show_json("Agent HTTP 호출 순서", timeline)

paths = [item["path"] for item in timeline]
checks = {
    "계좌 확인 API 1회": paths.count("/api/v1/agent-tools/accounts") == 1,
    "Prepare API 1회": (
        paths.count("/api/v1/agent-tools/settings/account-alias:prepare") == 1
    ),
    "실행 API 1회": paths.count("/api/v1/agent-tools/settings/account-alias") == 1,
    "결과 component Webhook 전송": timeline[-1].get("event_type") == "component",
    "Workflow 완료": completed.status == "completed",
}
show_json("자동 확인 결과", checks)
assert all(checks.values())


--- Agent HTTP 호출 순서 ---
[
  {
    "method": "GET",
    "path": "/api/v1/agent-tools/accounts",
    "request_id": "req_alias_start_001:resolve_account_alias_target",
    "execution_context_id": "exec_testbed_account_alias",
    "query_keys": [
      "account_capability",
      "account_hint",
      "all_accounts_requested",
      "limit",
      "resolve_selection"
    ]
  },
  {
    "method": "POST",
    "path": "/api/v1/agent-tools/settings/account-alias:prepare",
    "request_id": "req_alias_start_001:prepare_account_alias_change",
    "execution_context_id": "exec_testbed_account_alias"
  },
  {
    "method": "POST",
    "path": "/api/v1/webhooks/agent",
    "request_id": "req_alias_start_001",
    "execution_context_id": "exec_testbed_account_alias",
    "event_type": "need_approval",
    "step_id": "request_account_alias_approval"
  },
  {
    "method": "POST",
    "path": "/api/v1/agent-tools/settings/account-alias",
    "request_id": "req_alias_resume_approval_001:execute_accou

## 8. 다른 Route는 무엇이 달라지는가

| 지점 | Backend 응답/사용자 선택 | 다음 동작 |
| --- | --- | --- |
| 계좌 확인 | `selection_required` | `request_account_alias_selection`(선택 화면) |
| 계좌 확인 | `no_accounts` | `emit_account_alias_selection_empty`(빈 상태, 종료) |
| 별칭 확인 | 발화에 별칭 없음 | `request_account_alias_input`(별칭 입력 화면) |
| Prepare | `unchanged` | `emit_account_alias_unchanged`(오류 아님, 종료) |
| Prepare/Execute | `correction_required` + `account` | `reset_account_alias_target` → 계좌 확인부터 재시작 |
| Prepare/Execute | `correction_required` + `alias` | `reset_account_alias_value` → 별칭 입력부터 재시작 |
| Prepare/Execute | `blocked` | `emit_account_alias_blocked`(현재 공용 `blocked` 이벤트 타입 갭으로 `component` 대체) |
| 승인 화면 | `change_requested` + `account`/`alias` | 해당 대상만 초기화 후 재시작 |
| 승인 화면 | `cancelled` | 추가 Webhook 없이 종료 |

자동 테스트(`tests/test_set_account_alias_reference_workflow.py`)가 이 경로를
전부 검증한다(정정 대상 2종 분리 포함).

## 9. 실제 Backend로 전환하기

아래 Cell은 기본적으로 실행하지 않는다.

In [13]:
RUN_REAL_BACKEND = False

if RUN_REAL_BACKEND:
    required_variables = [
        "BACKEND_BASE_URL",
        "AGENT_SERVICE_TOKEN",
        "AGENT_WEBHOOK_SECRET",
        "TESTBED_CHAT_SESSION_ID",
        "TESTBED_EXECUTION_CONTEXT_ID",
    ]
    missing = [name for name in required_variables if not os.getenv(name)]
    if missing:
        raise RuntimeError(f"Backend Mode 환경변수가 없습니다: {missing}")

    real_config = BackendClientConfig(
        base_url=os.environ["BACKEND_BASE_URL"],
        agent_service_token=SecretStr(os.environ["AGENT_SERVICE_TOKEN"]),
        agent_webhook_secret=SecretStr(os.environ["AGENT_WEBHOOK_SECRET"]),
    )
    real_testbed = create_account_alias_change_backend_testbed(
        real_config, thread_id="thread_account_alias_real_001"
    )
    real_start = await real_testbed.start(
        message=USER_MESSAGE,
        request_id="req_alias_real_001",
        chat_session_id=os.environ["TESTBED_CHAT_SESSION_ID"],
        execution_context_id=os.environ["TESTBED_EXECUTION_CONTEXT_ID"],
    )
    show_json("실제 Backend 시작 결과", real_start.model_dump(mode="json"))
    await real_testbed.aclose()
else:
    print("Mock Backend Mode — 실제 Backend 호출 없음")

Mock Backend Mode — 실제 Backend 호출 없음


## 10. Testbed 종료

In [14]:
await testbed.aclose()
print("계좌 별칭 변경 단계별 Testbed 종료 완료")

계좌 별칭 변경 단계별 Testbed 종료 완료
